In [ ]:
!pip install -q kagglehub openpyxl nltk scikit-learn tensorflow pandas numpy scipy

import os
import re
import json
import glob
import random
import pickle
import kagglehub
import numpy as np
import pandas as pd
import nltk
from scipy import sparse
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Kunci random seed agar 100% reproducible (Bab 10.3)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Unduh daftar stopwords Bahasa Indonesia
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
STOPWORDS_ID = set(stopwords.words('indonesian'))

os.makedirs("data/processed", exist_ok=True)

# 2. Load data 
if os.path.exists("data/merged_dataset_eda.csv"):
    df = pd.read_csv("data/merged_dataset_eda.csv")
    print("Memuat dari data/merged_dataset_eda.csv")
else:
    dataset_dir = kagglehub.dataset_download("linkgish/indonesian-fact-and-hoax-political-news")
    all_xlsx = glob.glob(os.path.join(dataset_dir, "**", "*.xlsx"), recursive=True)
    target_files = [f for f in all_xlsx if "cleaned" in os.path.basename(f).lower()] or all_xlsx
    dfs = []
    for file_path in sorted(target_files):
        fname = os.path.basename(file_path).lower()
        temp = pd.read_excel(file_path)
        cols_lower = {c.lower(): c for c in temp.columns}
        title_col = next((cols_lower[k] for k in ['title', 'judul'] if k in cols_lower), None)
        text_col = next((cols_lower[k] for k in ['text_new', 'clean narasi', 'fulltext', 'narasi', 'isi', 'text'] if k in cols_lower), None)
        label_col = next((cols_lower[k] for k in ['hoax', 'label'] if k in cols_lower), None)

        std_df = pd.DataFrame()
        std_df['source'] = fname.replace('dataset_', '').replace('_cleaned.xlsx', '').replace('.xlsx', '')
        std_df['title'] = temp[title_col].astype(str) if title_col else ""
        std_df['body'] = temp[text_col].astype(str) if text_col else ""
        std_df['label'] = pd.to_numeric(temp[label_col], errors='coerce') if label_col else (1 if 'turnbackhoax' in fname else 0)
        std_df['label'] = std_df['label'].fillna(1 if 'turnbackhoax' in fname else 0).astype(int)
        dfs.append(std_df)
    df = pd.concat(dfs, ignore_index=True)
    df['raw_text'] = (df['title'].fillna('') + ' ' + df['body'].fillna('')).str.strip()

print(f"Ukuran data awal: {df.shape}")

100%|██████████| 101M/101M [00:03<00:00, 35.1MB/s] 

Extracting files...


Ukuran data awal: (31353, 5)


In [ ]:
# A. Hapus nilai kosong & string 'nan'
df['raw_text'] = df['raw_text'].replace({'nan': np.nan, 'nan nan': np.nan, '': np.nan})
before_drop = len(df)
df = df.dropna(subset=['raw_text', 'label']).copy()
print(f"Hapus missing values: {before_drop - len(df)} baris dibuang. Sisa: {len(df)}")

# B. Hapus duplikat
before_dup = len(df)
df = df.drop_duplicates(subset=['raw_text']).copy()
print(f"Hapus duplikat: {before_dup - len(df)} baris dibuang. Sisa: {len(df)}")

# C. Daftar kata bocoran media/format (agar model tidak curang menghafal nama portal!)
MEDIA_LEAK_WORDS = [
    r'\bcnn indonesia\b', r'\bcnnindonesia\b', r'\bcnn\b',
    r'\bkompas\.com\b', r'\bkompas\b',
    r'\btempo\.co\b', r'\btempo\b',
    r'\bturnbackhoax\.id\b', r'\bturnbackhoax\b', r'\bmafindo\b',
    r'\bsalah\b', r'\bpenjelasan\b', r'\bnarasi\b', r'\breferensi\b',
    r'\bhasil periksa fakta\b', r'\bperiksa fakta\b', r'\bbaca juga\b',
    r'\bpewarta\b', r'\beditor\b', r'\bjakarta\b'
]

# D. Daftar entitas tokoh/instansi politik populer (Mitigasi Keyword Overfitting di Bab 2.2)
POLITICAL_ENTITIES = [
    r'\bjoko widodo\b', r'\bjokowi\b', r'\bprabowo subianto\b', r'\bprabowo\b',
    r'\banies baswedan\b', r'\banies\b', r'\bganjar pranowo\b', r'\bganjar\b',
    r'\bgibran\b', r'\bmegawati\b', r'\broy suryo\b', r'\bmahfud\b',
    r'\bugm\b', r'\bpdip\b', r'\bgerindra\b', r'\bgolkar\b', r'\bnasdem\b'
]

def clean_and_mask_text(text):
    text = str(text).lower()
    # 1. Hapus URL, mention, hashtag, dan tag HTML
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'@\w+|#\w+', ' ', text)
    # 2. Hapus penanda tag kurung siku seperti [SALAH], [HOAKS], [KLARIFIKASI]
    text = re.sub(r'\[.*?\]', ' ', text)
    # 3. Hapus kata bocoran sumber media
    for pattern in MEDIA_LEAK_WORDS:
        text = re.sub(pattern, ' ', text)
    # 4. Samarkan nama tokoh/instansi politik agar model fokus ke pola bahasa
    for pattern in POLITICAL_ENTITIES:
        text = re.sub(pattern, ' tokohpublik ', text)
    # 5. Hapus angka dan tanda baca (sisakan huruf a-z dan spasi)
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 6. Tokenisasi sederhana + hapus stopwords Bahasa Indonesia & kata < 2 huruf
    tokens = [w for w in text.split() if w not in STOPWORDS_ID and len(w) > 2]
    return " ".join(tokens)

df['clean_text'] = df['raw_text'].apply(clean_and_mask_text)
df['clean_word_count'] = df['clean_text'].apply(lambda x: len(x.split()))

# E. Tangani outlier teks terlalu pendek (< 5 kata setelah cleaning) & duplikat baru pasca-cleaning
before_short = len(df)
df = df[df['clean_word_count'] >= 5].drop_duplicates(subset=['clean_text']).copy()
df = df.reset_index(drop=True)
print(f"Hapus teks terlalu pendek (<5 kata) & duplikat pasca-cleaning: {before_short - len(df)} baris dibuang.")
print(f"TOTAL DATA BERSIH SIAP SPLIT: {len(df)} baris")
print("\nDistribusi Label Pasca-Cleaning:")
print(df['label'].value_counts())

# Tampilkan contoh perbandingan sebelum vs sesudah cleaning 
display(df[['source', 'label', 'raw_text', 'clean_text']].head(3))

Hapus missing values: 21 baris dibuang. Sisa: 31332
Hapus duplikat: 14 baris dibuang. Sisa: 31318
Hapus teks terlalu pendek (<5 kata) & duplikat pasca-cleaning: 409 baris dibuang.
TOTAL DATA BERSIH SIAP SPLIT: 30909 baris

Distribusi Label Pasca-Cleaning:
label
0    20949
1     9960
Name: count, dtype: int64


,source,label,raw_text,clean_text
0,NaN,0,Anies di Milad BKMT: Pengajian Menghasilkan Ib...,tokohpublik milad bkmt pengajian menghasilkan ...
1,NaN,0,Edy Soal Pilgub Sumut: Kalau yang Maju Abal-ab...,edy pilgub sumut maju abal abal terpaksa maju ...
2,NaN,0,PKB Bakal Daftarkan Menaker Ida Fauziyah Jadi ...,pkb daftarkan menaker ida fauziyah caleg dpr p...


In [3]:
# Tahap 1: Pisahkan 60% Training dan 40% Temporary (Val + Test)
X_train_text, X_temp_text, y_train, y_temp = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.40,
    random_state=SEED,
    stratify=df['label']
)

# Tahap 2: Bagi 40% Temporary menjadi 20% Validation dan 20% Test
X_val_text, X_test_text, y_val, y_test = train_test_split(
    X_temp_text,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

# Cetak tabel jumlah sampel tiap split (WAJIB dicatat di Bab 5.4 & Checklist LKM11!)
split_summary = pd.DataFrame({
    'Split': ['Training (60%)', 'Validation (20%)', 'Test (20%)'],
    'Total Sampel': [len(X_train_text), len(X_val_text), len(X_test_text)],
    'Kelas 0 (Fakta)': [(y_train == 0).sum(), (y_val == 0).sum(), (y_test == 0).sum()],
    'Kelas 1 (Hoaks)': [(y_train == 1).sum(), (y_val == 1).sum(), (y_test == 1).sum()],
    'Rasio Hoaks (%)': [
        round((y_train == 1).mean() * 100, 2),
        round((y_val == 1).mean() * 100, 2),
        round((y_test == 1).mean() * 100, 2)
    ]
})
display(split_summary)

,Split,Total Sampel,Kelas 0 (Fakta),Kelas 1 (Hoaks),Rasio Hoaks (%)
0,Training (60%),18545,12569,5976,32.22
1,Validation (20%),6182,4190,1992,32.22
2,Test (20%),6182,4190,1992,32.22


In [4]:
# =========================================================================
# 1. TRANSFORMASI TF-IDF (UNTUK MODEL 1: LOGISTIC REGRESSION & MODEL 2: SVM)
# ATURAN EMAS: fit_transform HANYA di X_train, transform SAJA di X_val & X_test!
# =========================================================================
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=3)

X_train_tfidf = tfidf.fit_transform(X_train_text)   # <--- FIT HANYA DI TRAIN
X_val_tfidf   = tfidf.transform(X_val_text)         # <--- TRANSFORM SAJA
X_test_tfidf  = tfidf.transform(X_test_text)        # <--- TRANSFORM SAJA (DISEGEL!)

print("Shape TF-IDF Train:", X_train_tfidf.shape)
print("Shape TF-IDF Val  :", X_val_tfidf.shape)
print("Shape TF-IDF Test :", X_test_tfidf.shape)

# =========================================================================
# 2. TRANSFORMASI TOKENIZER & PADDING (UNTUK MODEL 3: NEURAL NETWORK)
# ATURAN EMAS: fit_on_texts HANYA di X_train!
# =========================================================================
VOCAB_SIZE = 10000
MAX_LEN = 150  # Panjang maksimum token per artikel (cukup menangkap judul & inti berita)

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)  # <--- FIT HANYA DI TRAIN

X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_text), maxlen=MAX_LEN, padding='post', truncating='post')
X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val_text),   maxlen=MAX_LEN, padding='post', truncating='post')
X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test_text),  maxlen=MAX_LEN, padding='post', truncating='post')

print("Shape Sequence Train:", X_train_seq.shape)
print("Shape Sequence Val  :", X_val_seq.shape)
print("Shape Sequence Test :", X_test_seq.shape)

# =========================================================================
# 3. DATA BALANCING HANYA PADA DATA TRAINING (CLASS WEIGHTS)
# =========================================================================
weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_train)
class_weight_dict = {0: round(float(weights[0]), 4), 1: round(float(weights[1]), 4)}
print("\nClass Weights (dihitung HANYA dari y_train):", class_weight_dict)

Shape TF-IDF Train: (18545, 10000)
Shape TF-IDF Val  : (6182, 10000)
Shape TF-IDF Test : (6182, 10000)
Shape Sequence Train: (18545, 150)
Shape Sequence Val  : (6182, 150)
Shape Sequence Test : (6182, 150)

Class Weights (dihitung HANYA dari y_train): {0: 0.7377, 1: 1.5516}


In [5]:
# 1. Simpan data teks bersih per split
pd.DataFrame({'clean_text': X_train_text, 'label': y_train}).to_csv("data/processed/train_clean.csv", index=False)
pd.DataFrame({'clean_text': X_val_text,   'label': y_val}).to_csv("data/processed/val_clean.csv", index=False)
pd.DataFrame({'clean_text': X_test_text,  'label': y_test}).to_csv("data/processed/test_clean.csv", index=False)

# 2. Simpan matriks TF-IDF (format .npz hemat memori)
sparse.save_npz("data/processed/X_train_tfidf.npz", X_train_tfidf)
sparse.save_npz("data/processed/X_val_tfidf.npz",   X_val_tfidf)
sparse.save_npz("data/processed/X_test_tfidf.npz",  X_test_tfidf)

# 3. Simpan urutan token Neural Network (format .npy)
np.save("data/processed/X_train_seq.npy", X_train_seq)
np.save("data/processed/X_val_seq.npy",   X_val_seq)
np.save("data/processed/X_test_seq.npy",  X_test_seq)

# 4. Simpan target label (y)
np.save("data/processed/y_train.npy", y_train.values)
np.save("data/processed/y_val.npy",   y_val.values)
np.save("data/processed/y_test.npy",  y_test.values)

# 5. Simpan objek Vectorizer, Tokenizer, dan Class Weights
with open("data/processed/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
with open("data/processed/tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)
with open("data/processed/class_weights.json", "w") as f:
    json.dump(class_weight_dict, f, indent=2)

print("SEMUA FILE OUTPUT MINGGU 2 BERHASIL DISIMPAN DI data/processed/!")

SEMUA FILE OUTPUT MINGGU 2 BERHASIL DISIMPAN DI data/processed/!
